In [54]:
import requests, os, zipfile, shutil
from pathlib import Path
import javalang
import pandas as pd
from dotenv import load_dotenv
import time
import re
import ast
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from typing import List, Dict, Iterable, Optional

# 1. Get python file

In [7]:
def search_python_repos(n=1, min_star=50, header={}, allowed_licenses=None):
    """
    Search Java repos by stars with pagination support.
    Filters by license if allowed_licenses is provided.
    """
    url = "https://api.github.com/search/repositories"
    language = "Python"
    
    repos, page = [], 1
    while len(repos) < n:
        params = {
            "q": f"language:{language} stars:>{min_star}",
            "sort": "stars",
            "order": "desc",
            "per_page": 100,
            "page": page,
        }
        r = requests.get(url, headers=header, params=params, timeout=30)
        r.raise_for_status()
        items = r.json().get("items", [])
        if not items:
            break

        for repo in items:
            lic = repo.get("license")
            if allowed_licenses:
                if not lic or lic["key"].lower() not in allowed_licenses:
                    continue  # skip incompatible/missing licenses
            repos.append(repo)
            if len(repos) >= n:
                break

        page += 1
    
    return repos[:n]

In [32]:
def get_repo_files(owner, repo, header, branch="main", extension=".py", max_files=None):
    """
    Get all file paths in a repo.
    * Arguments:
        - owner: repo owner
        - repo: repo name
        - branch: branch name (default: main)
    * Returns:
        - list of file paths (strings)
    """

    url = f"https://api.github.com/repos/{owner}/{repo}/git/trees/{branch}?recursive=1"
    r = requests.get(url, headers=header, timeout=30)
    r.raise_for_status()
    tree = r.json().get("tree", [])
    
    result = [item["path"] for item in tree if item["type"] == "blob" and item["path"].endswith(extension)]
    if max_files:
        result = result[:max_files]
    
    return result

In [33]:
def download_and_extract_repo(owner, repo, header, branch, dest):
    """Download repo as zipball and extract it locally."""
    dest.mkdir(parents=True, exist_ok=True)
    zip_path = dest / f"{owner}-{repo}-{branch}.zip"
    extract_dir = dest / f"{owner}-{repo}-{branch}"

    if extract_dir.exists():
        shutil.rmtree(extract_dir)

    url = f"https://api.github.com/repos/{owner}/{repo}/zipball/{branch}"
    r = requests.get(url, headers=header, stream=True, timeout=60)
    r.raise_for_status()

    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(1024 * 256):
            f.write(chunk)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)

    # GitHub zip includes a single top-level folder, return that
    subfolders = list(extract_dir.iterdir())
    return subfolders[0] if subfolders else extract_dir

In [60]:
def deduplicate_methods(df):
    """Remove duplicate methods from the dataset."""
    return df.drop_duplicates(
        subset=["repo_name", "file_path", "method_code"]
    )

def extract_python_methods(file_path):
    """Extract all Python method definitions from a file."""
    methods = []
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        code = f.read()
    try:
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                methods.append(ast.get_source_segment(code, node))
    except (javalang.parser.JavaSyntaxError, IndexError):
        pass  # skip files with syntax errors
    return methods


def _worker(args) -> Optional[List[Dict]]:
    """
    Worker run in a separate process.
    Returns a list of result dicts for one file, or None if the file is skipped/errored.
    """
    file_path, local_repo_path, repo_name, repo_url = args
    full_path = os.path.join(local_repo_path, file_path)
    try:
        if not os.path.isfile(full_path):
            return None

        methods = extract_python_methods(full_path)  # must be importable / top-level
        if not methods:
            return []

        out = []
        for method in methods:
            out.append({
                "repo_name": repo_name,
                "repo_url": repo_url,
                "file_path": file_path,
                "method_code": method,
            })
        return out
    except Exception as e:
        return None
    
def extract_python_method_parallel(
    files: list,
    local_repo_path: str,
    repo_name: str,
    repo_url: str,
    max_workers: int | None = None,
    chunksize: int = 32,
) -> List[Dict]:
    """
    Parallelize over files using processes (good for CPU-bound AST parsing).
    """
    all_results: List[Dict] = []
    tasks = ((fp, local_repo_path, repo_name, repo_url) for fp in files)

    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        for res in ex.map(_worker, tasks, chunksize=chunksize):
            if res:  # None or [] are both fine; [] adds nothing
                all_results.extend(res)

    return all_results

In [61]:
load_dotenv()
TOKEN = os.getenv("GITHUB_TOKEN")
if not TOKEN:
    raise SystemExit("❌ Please set GITHUB_TOKEN in your environment.")

HEADERS = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/vnd.github+json"}



DATA_DIR_RAW = Path("/home/tnguyen10/Desktop/WM/AI-and-SE/assignment_1/dataset/raw") 

OUTPUT_RAW_CSV = DATA_DIR_RAW / "python_methods_raw.csv"

N_REPOS = 10
MIN_STAR_REPO = 50
MAX_FILE_PER_REPO = 1_000
ALLOWED_LICENSES = {"mit", "apache-2.0", "bsd-2-clause", "bsd-3-clause"}
FILE_EXTENSION = ".py"

In [62]:
repos = search_python_repos(n=N_REPOS, min_star=MIN_STAR_REPO, header=HEADERS,\
            allowed_licenses=ALLOWED_LICENSES)

print(f"Found {len(repos)} Python repos.")

Found 10 Python repos.


In [ ]:
all_results = []

for idx, repo in enumerate(repos):
    print(f"\nProcessing repo: {repo['full_name']} (branch={branch})")
    
    try:
        owner, name = repo["full_name"].split("/")
        branch = repo["default_branch"]
        repo_url = repo["html_url"]
        
        local_repo_path = download_and_extract_repo(owner, name, HEADERS, branch, DATA_DIR_RAW)
        print(f"   Downloaded and extracted to {local_repo_path}")
        
        files = get_repo_files(owner, name, HEADERS, branch,\
                        extension=FILE_EXTENSION, max_files=MAX_FILE_PER_REPO)
        print(f"   Found {len(files)} .py files")
        
        current_repo_results = extract_python_method_parallel(
            files,
            str(local_repo_path),
            repo["full_name"],
            repo_url,
            max_workers=8,
            chunksize=16,
        )
        
        all_results.extend(current_repo_results)
        
    except Exception as e:
        print(f"[ERROR] Failed to process repo {repo['full_name']}: {e}")
        continue


Processing repo: public-apis/public-apis (branch=master)


   Downloaded and extracted to /home/tnguyen10/Desktop/WM/AI-and-SE/assignment_1/dataset/raw/public-apis-public-apis-master/public-apis-public-apis-675a4e6
   Found 6 .py files

Processing repo: TheAlgorithms/Python (branch=master)
   Downloaded and extracted to /home/tnguyen10/Desktop/WM/AI-and-SE/assignment_1/dataset/raw/TheAlgorithms-Python-master/TheAlgorithms-Python-3b08413
   Found 1000 .py files

Processing repo: huggingface/transformers (branch=master)
   Downloaded and extracted to /home/tnguyen10/Desktop/WM/AI-and-SE/assignment_1/dataset/raw/huggingface-transformers-main/huggingface-transformers-307c523
   Found 1000 .py files

Processing repo: langflow-ai/langflow (branch=main)
   Downloaded and extracted to /home/tnguyen10/Desktop/WM/AI-and-SE/assignment_1/dataset/raw/langflow-ai-langflow-main/langflow-ai-langflow-621c103
   Found 1000 .py files

Processing repo: langchain-ai/langchain (branch=main)
   Downloaded and extracted to /home/tnguyen10/Desktop/WM/AI-and-SE/assignm

In [ ]:
df = pd.DataFrame(all_results)
df = deduplicate_methods(df)    
df = df.reset_index(drop=True)

df.to_csv(OUTPUT_RAW_CSV, index=False)
print("=================================")
print(f"[DONE] Dataset saved to {OUTPUT_RAW_CSV}")
print(f"Total samples: {len(df)}")

[DONE] Dataset saved to /home/tnguyen10/Desktop/WM/AI-and-SE/assignment_1/dataset/raw/python_methods_raw.csv, total samples: 35997


In [67]:
idx = np.random.randint(0, len(df))
print(df.loc[idx, "method_code"])

def lc_id(cls) -> list[str]:
        """Return a unique identifier for this class for serialization purposes.

        The unique identifier is a list of strings that describes the path
        to the object.

        For example, for the class `langchain.llms.openai.OpenAI`, the id is
        `["langchain", "llms", "openai", "OpenAI"]`.
        """
        # Pydantic generics change the class name. So we need to do the following
        if (
            "origin" in cls.__pydantic_generic_metadata__
            and cls.__pydantic_generic_metadata__["origin"] is not None
        ):
            original_name = cls.__pydantic_generic_metadata__["origin"].__name__
        else:
            original_name = cls.__name__
        return [*cls.get_lc_namespace(), original_name]
